# COMP9312 Project Q2: k-triangle core index


Run the cells from top to bottom. Only edit the `KTriangleIndex` code cell.


## 1. Graph Data Structure
The following code defines the input graph representation. Run this cell first. Do not edit this cell.


In [ ]:
class Graph:
    def __init__(self, n, edges):
        """
        Build an undirected graph using a compact adjacency-array structure.

        Parameters
        ----------
        n : int
            Number of vertices. Vertices are assumed to be numbered from 0 to n - 1.

        edges : list[tuple[int, int]]
            List of undirected edges. For example, (u, v) means there is an edge
            between vertex u and vertex v.
        """

        # Number of vertices
        self.n = n

        # Number of undirected edges
        self.m = len(edges)

        # degree[v] stores the degree of vertex v
        degree = [0] * n

        # Count the degree of each vertex
        for u, v in edges:
            degree[u] += 1
            degree[v] += 1

        # offsets[v] stores the starting position of vertex v's neighbors
        # in the indices array.
        #
        # The neighbors of vertex v are stored in:
        #
        # indices[offsets[v] : offsets[v + 1]]
        self.offsets = [0] * (n + 1)

        # Build prefix sums from the degree array
        for v in range(n):
            self.offsets[v + 1] = self.offsets[v] + degree[v]

        # indices stores all adjacency lists in one flat array.
        # Since the graph is undirected, each edge is stored twice:
        # u -> v and v -> u.
        self.indices = [0] * (2 * self.m)

        # cursor[v] points to the next free position in vertex v's adjacency area
        cursor = self.offsets[:-1].copy()

        # Fill the adjacency array
        for u, v in edges:
            self.indices[cursor[u]] = v
            cursor[u] += 1

            self.indices[cursor[v]] = u
            cursor[v] += 1

        # Sort each vertex's adjacency list.
        # This makes edge lookup efficient using binary search.
        for v in range(n):
            start = self.offsets[v]
            end = self.offsets[v + 1]
            self.indices[start:end] = sorted(self.indices[start:end])

    def neighbors(self, v):
        """
        Return all neighbors of vertex v.

        Example
        -------
        If vertex 1 is connected to 0, 2, and 3,
        then neighbors(1) returns [0, 2, 3].
        """

        start = self.offsets[v]
        end = self.offsets[v + 1]
        return self.indices[start:end]

    def degree(self, v):
        """
        Return the degree of vertex v.

        Example
        -------
        If vertex 1 has three neighbors,
        then degree(1) returns 3.
        """

        return self.offsets[v + 1] - self.offsets[v]

    def has_edge(self, u, v):
        """
        Return True if there is an edge between u and v.

        The adjacency list of u is sorted, so binary search can be used.

        Example
        -------
        If the graph contains edge (0, 2), then has_edge(0, 2) returns True.
        If the graph does not contain edge (0, 4), then has_edge(0, 4) returns False.
        """

        left = self.offsets[u]
        right = self.offsets[u + 1] - 1

        while left <= right:
            mid = (left + right) // 2

            if self.indices[mid] == v:
                return True
            elif self.indices[mid] < v:
                left = mid + 1
            else:
                right = mid - 1

        return False

## 2. Code Template
Only edit this cell. Implement `KTriangleIndex.build()` and `KTriangleIndex.query(k, v)`.


In [ ]:
################################################################################
# Import Python Standard Library modules here if needed.
from typing import List
################################################################################


class KTriangleIndex:
    """K-triangle core index.

    Students should implement this class. The input graph has already been
    loaded as Graph and is available as self.graph.

    Add helper methods and fields inside this class as needed, but do not change
    the public signatures of build() and query().
    """

    def __init__(self, graph: "Graph"):
        self.graph = graph
        self.tc = None          # triangle-core number of each vertex
        self.level = None       # per-slot edge level, parallel to graph.indices

    def build(self) -> None:
        """Build the k-triangle core index.

        This method is called once before any query is evaluated.
        """
        g = self.graph
        n = g.n
        off, adj = g.offsets, g.indices
        deg = [off[v + 1] - off[v] for v in range(n)]

        # Orient each edge from the lower to the higher (degree, id). Then every
        # triangle shows up exactly once, seen from its lowest-ranked vertex.
        fwd = [[] for _ in range(n)]
        for u in range(n):
            ku = (deg[u], u)
            for i in range(off[u], off[u + 1]):
                w = adj[i]
                if ku < (deg[w], w):
                    fwd[u].append(w)

        triangles = []
        tcount = [0] * n
        seen = [-1] * n
        for u in range(n):
            for w in fwd[u]:
                seen[w] = u
            for w in fwd[u]:
                for x in fwd[w]:
                    if seen[x] == u:            # edge u-x present -> triangle
                        triangles.append((u, w, x))
                        tcount[u] += 1
                        tcount[w] += 1
                        tcount[x] += 1

        tri_of = [[] for _ in range(n)]
        for t, (a, b, c) in enumerate(triangles):
            tri_of[a].append(t)
            tri_of[b].append(t)
            tri_of[c].append(t)

        self.tc = self._peel(tcount, triangles, tri_of)
        self.level = self._edge_levels(triangles)

    def _peel(self, tcount, triangles, tri_of):
        # k-core peeling, but on triangle counts instead of degrees: repeatedly
        # drop the vertex in fewest triangles; killing it removes those triangles
        # from its two partners. Batagelj-Zaversnik buckets keep every step O(1).
        n = len(tcount)
        d = tcount[:]
        top = max(d) if n else 0

        head = [0] * (top + 2)
        for v in range(n):
            head[d[v]] += 1
        pos_start = 0
        for c in range(top + 1):
            head[c], pos_start = pos_start, pos_start + head[c]
        order = [0] * n
        pos = [0] * n
        for v in range(n):
            pos[v] = head[d[v]]
            order[pos[v]] = v
            head[d[v]] += 1
        for c in range(top, 0, -1):
            head[c] = head[c - 1]
        head[0] = 0

        alive = bytearray([1]) * len(triangles)
        tc = [0] * n
        for i in range(n):
            v = order[i]
            lvl = d[v]
            tc[v] = lvl
            for t in tri_of[v]:
                if not alive[t]:
                    continue
                alive[t] = 0
                for w in triangles[t]:
                    dw = d[w]
                    if w == v or dw <= lvl:     # w already fixed, leave it
                        continue
                    # move w one bucket down
                    p, s = pos[w], head[dw]
                    f = order[s]
                    if f != w:
                        order[p], order[s] = f, w
                        pos[f], pos[w] = p, s
                    head[dw] += 1
                    d[w] = dw - 1
        return tc

    def _edge_levels(self, triangles):
        g = self.graph
        off, adj, tc = g.offsets, g.indices, self.tc

        # An edge's level is the best min(tc) over the triangles it belongs to.
        best = {}
        for a, b, c in triangles:
            lvl = min(tc[a], tc[b], tc[c])
            for x, y in ((a, b), (a, c), (b, c)):
                key = (x, y) if x < y else (y, x)
                if best.get(key, -1) < lvl:
                    best[key] = lvl

        level = [0] * (2 * g.m)
        for u in range(g.n):
            for i in range(off[u], off[u + 1]):
                w = adj[i]
                key = (u, w) if u < w else (w, u)
                if key in best:
                    level[i] = best[key]
        return level

    def query(self, k: int, v: int) -> List[int]:
        """Return the indexed k-triangle component containing v.

        The return value must be a Python list of vertex IDs.

        Example return value:

            [0, 1, 2, 3, 4, 5]
        """
        if self.tc[v] < k:
            return []

        off, adj, level = self.graph.offsets, self.graph.indices, self.level
        seen = {v}
        stack = [v]
        while stack:
            x = stack.pop()
            for i in range(off[x], off[x + 1]):
                if level[i] >= k and adj[i] not in seen:
                    seen.add(adj[i])
                    stack.append(adj[i])
        return sorted(seen)

## 3. How to Test Your Code
The following tests use the `KTriangleIndex` class defined above, generate fixed-seed graphs, and compare your output with precomputed answers. The largest test has 10,000 vertices and 80,000 edges.


In [ ]:
################################################################################
# Do not edit this code cell.
import time
from itertools import accumulate, combinations
################################################################################


def generate_test_graph(seed, num_cliques=4):
    rng = _LCG(seed)
    clique_sizes = [rng.randint(4, 6) for _ in range(num_cliques)]
    prefix = [0] + list(accumulate(clique_sizes))
    edges = set()

    for idx, size in enumerate(clique_sizes):
        base = prefix[idx]
        for i, j in combinations(range(size), 2):
            edges.add((base + i, base + j))

    for idx in range(num_cliques - 1):
        u = prefix[idx] + clique_sizes[idx] - 1
        v = prefix[idx + 1]
        if u > v:
            u, v = v, u
        edges.add((u, v))

    next_id = prefix[-1]
    for _ in range(num_cliques):
        clique_id = rng.randint(0, num_cliques - 1)
        anchor = prefix[clique_id] + rng.randint(0, clique_sizes[clique_id] - 1)
        a, b = next_id, next_id + 1
        next_id += 2
        for u, v in ((anchor, a), (anchor, b), (a, b)):
            if u > v:
                u, v = v, u
            edges.add((u, v))

    return Graph(next_id, sorted(edges))


def generate_benchmark_graph(seed=9312):
    n = 10000
    edges = set()

    for i, j in combinations(range(0, 6), 2):
        edges.add((i, j))
    for i, j in combinations(range(6, 11), 2):
        edges.add((i, j))
    edges.add((5, 6))
    edges.add((10, 11))

    rng = _LCG(seed)
    left_start, left_end = 11, 5006
    right_start, right_end = 5006, 10000
    right_count = right_end - right_start

    for offset, u in enumerate(range(left_start, left_end)):
        shift = rng.randint(0, right_count - 1)
        for j in range(16):
            v = right_start + ((offset + shift + j) % right_count)
            edges.add((u, v))

    extra_edges_needed = 80000 - len(edges)
    added = 0
    cursor = 0
    while added < extra_edges_needed:
        u = left_start + (cursor % (left_end - left_start))
        v = right_start + ((cursor * 37 + 16) % right_count)
        edge = (u, v)
        if edge not in edges:
            edges.add(edge)
            added += 1
        cursor += 1

    return Graph(n, sorted(edges))


class _LCG:
    _a = 6364136223846793005
    _c = 1442695040888963407
    _m = 2 ** 64

    def __init__(self, seed=42):
        self.state = seed & 0xFFFFFFFFFFFFFFFF

    def rand64(self):
        self.state = (self._a * self.state + self._c) % self._m
        return self.state

    def randint(self, lo, hi):
        return lo + self.rand64() % (hi - lo + 1)


def run_tests() -> None:
    test_cases = [
        {
            "seed": 7,
            "generator": "small",
            "n": 30,
            "m": 66,
            "queries": [(1, 0), (3, 0), (3, 6), (4, 0), (1, 29), (10, 0)],
            "answers": [
                [0, 1, 2, 3, 4, 5, 24, 25, 28, 29],
                [0, 1, 2, 3, 4, 5],
                [6, 7, 8, 9],
                [0, 1, 2, 3, 4, 5],
                [0, 1, 2, 3, 4, 5, 24, 25, 28, 29],
                [0, 1, 2, 3, 4, 5],
            ],
        },
        {
            "seed": 42,
            "generator": "small",
            "n": 31,
            "m": 70,
            "queries": [(1, 0), (3, 0), (3, 6), (4, 0), (1, 30), (10, 0)],
            "answers": [
                [0, 1, 2, 3, 4, 5],
                [0, 1, 2, 3, 4, 5],
                [6, 7, 8, 9, 10, 11],
                [0, 1, 2, 3, 4, 5],
                [18, 19, 20, 21, 22, 25, 26, 29, 30],
                [0, 1, 2, 3, 4, 5],
            ],
        },
        {
            "seed": 2026,
            "generator": "small",
            "n": 28,
            "m": 56,
            "queries": [(1, 0), (3, 0), (3, 5), (4, 0), (1, 27), (10, 0)],
            "answers": [
                [0, 1, 2, 3, 4],
                [0, 1, 2, 3, 4],
                [5, 6, 7, 8, 9],
                [0, 1, 2, 3, 4],
                [16, 17, 18, 19, 22, 23, 26, 27],
                [],
            ],
        },
        {
            "seed": 9312,
            "generator": "benchmark",
            "n": 10000,
            "m": 80000,
            "queries": [(1, 0), (1, 7), (6, 7), (7, 7), (10, 0), (1, 11), (1, 9999)],
            "answers": [
                [0, 1, 2, 3, 4, 5],
                [6, 7, 8, 9, 10],
                [6, 7, 8, 9, 10],
                [],
                [0, 1, 2, 3, 4, 5],
                [],
                [],
            ],
        },
    ]

    for case in test_cases:
        if case["generator"] == "benchmark":
            graph = generate_benchmark_graph(case["seed"])
        else:
            graph = generate_test_graph(case["seed"])

        print(f"\nTesting {case['generator']} seed={case['seed']} | n={graph.n}, m={graph.m}")
        if graph.n != case["n"] or graph.m != case["m"]:
            print(f"Graph size mismatch: expected n={case['n']}, m={case['m']}")
            continue

        index = KTriangleIndex(graph)
        build_start = time.perf_counter()
        index.build()
        build_time = time.perf_counter() - build_start
        print(f"build time: {build_time:.6f} seconds")

        all_correct = True
        query_time_total = 0.0
        for query, expected in zip(case["queries"], case["answers"]):
            k, v = query
            query_start = time.perf_counter()
            actual = sorted(index.query(k, v))
            query_time = time.perf_counter() - query_start
            query_time_total += query_time
            ok = actual == expected
            all_correct = all_correct and ok
            status = "CORRECT" if ok else "INCORRECT"
            print(f"query({k}, {v}) -> {actual} | expected {expected} | {status} | time {query_time:.6f}s")
        print(f"total query time: {query_time_total:.6f} seconds")
        print("Result:", "CORRECT" if all_correct else "INCORRECT")

run_tests()
